# Stacking dos Top-5 Modelos Individuais (Patch-Level Only)

Gera predições dos 5 melhores modelos **patch-level** (sem MIL) e treina meta-classificadores
(Regressão Logística e Random Forest) sobre as probabilidades geradas.

| # | Modelo | QWK (teste) | Checkpoint |
|---|--------|-------------|------------|
| 1 | EfficientNetV2-S + Optuna (patch) | 0.8742 | `v2-optuna-ordinal-focal.pth` |
| 2 | EfficientNet-B0 (patch, melhor run) | 0.8589 | `b0-entropy-ordinal-3-focal.pth` |
| 3 | EfficientNet-B0 + SWA | 0.8523 | `b0-entropy-ordinal-swa.pth` |
| 4 | EfficientNetV2-S (sem Optuna) | 0.8521 | `efficientnet-v2-s-entropy-ordinal-focal.pth` |
| 5 | EfficientNet-B3 + Ordinal + Focal | 0.8484 | `b3-entropy-ordinal-focal.pth` |

**Estratégia anti-leakage:** Os modelos base foram treinados nos folds 0–2 e 4.  
O fold 3 (nunca visto pelo treino) serve como `X_meta_train`.  
O conjunto de teste serve como `X_meta_test`.

## 1. Imports

In [ ]:
import os
import sys
import gc
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import DataLoader, SequentialSampler
from torchvision.models import (
    efficientnet_b0,  EfficientNet_B0_Weights,
    efficientnet_b3,  EfficientNet_B3_Weights,
    efficientnet_v2_s, EfficientNet_V2_S_Weights,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score,
    f1_score, recall_score, precision_score,
    classification_report, confusion_matrix,
)
from sklearn.preprocessing import StandardScaler

sys.path.append('../..')
from utils.dataset import PandasDataset
from utils.models import EfficientNetApi

warnings.filterwarnings('ignore')
print('Imports OK')

## 2. Configuração

In [ ]:
SEED           = 42
NUM_WORKERS    = 4
OUTPUT_CLASSES = 5      # limiares ordinais ISUP 0-5
BATCH_SIZE     = 4
N_BOOT         = 1000
META_CV_FOLDS  = 5      # k-fold interno no meta-treino (fold 3)

# Hiperparâmetros dos checkpoints (conforme treinados)
DROPOUT_B0       = 0.6
DROPOUT_B3       = 0.6
DROPOUT_V2S_OPT  = 0.4422  # Optuna
UNFREEZE_V2S_OPT = 3       # Optuna: unfreeze_blocks
DROPOUT_V2S_PLAIN  = 0.4   # V2S sem HPO — verificar contra notebook de treino
UNFREEZE_V2S_PLAIN = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

# Caminhos
ROOT_DIR   = '../..'
DATA_DIR   = '../../..'
IMAGES_DIR = os.path.join(DATA_DIR, 'tiles')

CKPT_V2S_OPT   = '../tests/baseline/models/v2-optuna-ordinal-focal.pth'
CKPT_B0        = '../tests/baseline/models/b0-entropy-ordinal-3-focal.pth'
CKPT_SWA       = '../tests/baseline/models/b0-entropy-ordinal-swa.pth'
CKPT_V2S_PLAIN = '../tests/baseline/models/efficientnet-v2-s-entropy-ordinal-focal.pth'
CKPT_B3        = '../tests/baseline/models/b3-entropy-ordinal-focal.pth'

os.makedirs('logs', exist_ok=True)

print(f'Device    : {DEVICE}')
print(f'IMAGES_DIR: {IMAGES_DIR}')

## 3. Definições de Modelos

In [ ]:
class EfficientNetV2Api(nn.Module):
    """Wrapper EfficientNetV2-S com cabeça Sequential (LayerNorm + Dropout + Linear).
    Usado pelos checkpoints treinados com Optuna."""

    def __init__(self, model: nn.Module, output_dimensions: int,
                 dropout_rate: float = 0.4, unfreeze_blocks: int = 3):
        super().__init__()
        self.model = model
        for param in self.model.parameters():
            param.requires_grad = False
        if hasattr(self.model, 'features') and unfreeze_blocks > 0:
            for block in self.model.features[-unfreeze_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True
        if isinstance(self.model.classifier, nn.Sequential):
            in_features = self.model.classifier[-1].in_features
        else:
            in_features = self.model.classifier.in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, output_dimensions),
        )

    def forward(self, x):
        x = self.model(x)
        if x.ndim == 4:
            x = x.mean(dim=[2, 3])
        return self.head(x)


class EfficientNetV2PlainApi(nn.Module):
    """Wrapper EfficientNetV2-S com cabeça Linear simples (fully_connected).
    Corresponde aos checkpoints treinados sem Optuna."""

    def __init__(self, model: nn.Module, output_dimensions: int,
                 dropout_rate: float = 0.4, unfreeze_blocks: int = 2):
        super().__init__()
        self.model = model
        for param in self.model.parameters():
            param.requires_grad = False
        if hasattr(self.model, 'features') and unfreeze_blocks > 0:
            for block in self.model.features[-unfreeze_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True
        if isinstance(self.model.classifier, nn.Sequential):
            in_features = self.model.classifier[-1].in_features
        else:
            in_features = self.model.classifier.in_features
        self.model.classifier = nn.Identity()
        self.dropout = nn.Dropout(dropout_rate)
        self.fully_connected = nn.Linear(in_features, output_dimensions)

    def forward(self, x):
        x = self.model(x)
        if x.ndim == 4:
            x = x.mean(dim=[2, 3])
        return self.fully_connected(self.dropout(x))

## 4. Carregamento dos Modelos Base

In [ ]:
# ── 1. EfficientNetV2-S + Optuna (patch) ─────────────────────────────
print('Carregando V2S + Optuna...')
backbone_v2s_opt = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
model_v2s_opt = EfficientNetV2Api(
    model=backbone_v2s_opt,
    output_dimensions=OUTPUT_CLASSES,
    dropout_rate=DROPOUT_V2S_OPT,
    unfreeze_blocks=UNFREEZE_V2S_OPT,
)
model_v2s_opt.load_state_dict(torch.load(CKPT_V2S_OPT, weights_only=True))
model_v2s_opt = model_v2s_opt.to(DEVICE).eval()
print('  ✓ V2S+Optuna')

# ── 2. EfficientNet-B0 (patch, melhor run) ────────────────────────────
print('Carregando B0 (patch)...')
backbone_b0 = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model_b0 = EfficientNetApi(
    model=backbone_b0,
    output_dimensions=OUTPUT_CLASSES,
    dropout_rate=DROPOUT_B0,
)
model_b0.load_state_dict(torch.load(CKPT_B0, weights_only=True))
model_b0 = model_b0.to(DEVICE).eval()
print('  ✓ B0')

# ── 3. EfficientNet-B0 + SWA ──────────────────────────────────────────
print('Carregando B0 + SWA...')
backbone_swa = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model_swa = EfficientNetApi(
    model=backbone_swa,
    output_dimensions=OUTPUT_CLASSES,
    dropout_rate=DROPOUT_B0,
)
model_swa.load_state_dict(torch.load(CKPT_SWA, weights_only=True))
model_swa = model_swa.to(DEVICE).eval()
print('  ✓ B0+SWA')

# ── 4. EfficientNetV2-S (sem Optuna) — cabeça fully_connected ─────────
print('Carregando V2S (sem Optuna)...')
backbone_v2s_plain = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
model_v2s_plain = EfficientNetV2PlainApi(
    model=backbone_v2s_plain,
    output_dimensions=OUTPUT_CLASSES,
    dropout_rate=DROPOUT_V2S_PLAIN,
    unfreeze_blocks=UNFREEZE_V2S_PLAIN,
)
model_v2s_plain.load_state_dict(torch.load(CKPT_V2S_PLAIN, weights_only=True))
model_v2s_plain = model_v2s_plain.to(DEVICE).eval()
print('  ✓ V2S-plain')

# ── 5. EfficientNet-B3 + Ordinal + Focal ─────────────────────────────
print('Carregando B3...')
backbone_b3 = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
model_b3 = EfficientNetApi(
    model=backbone_b3,
    output_dimensions=OUTPUT_CLASSES,
    dropout_rate=DROPOUT_B3,
)
model_b3.load_state_dict(torch.load(CKPT_B3, weights_only=True))
model_b3 = model_b3.to(DEVICE).eval()
print('  ✓ B3')

print('\nTodos os 5 modelos patch-level carregados.')

## 5. Carregamento dos Dados

In [ ]:
df_all     = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
df_test    = pd.read_csv(f'{ROOT_DIR}/data/test.csv')

df_all.columns     = df_all.columns.str.strip()
df_entropy.columns = df_entropy.columns.str.strip()
df_test.columns    = df_test.columns.str.strip()

# Filtragem por entropia: remove top-20% mais difíceis (mesmo critério do treino)
hard_ids = set(
    df_entropy.sort_values('difficulty_score', ascending=False)
              .head(int(len(df_entropy) * 0.2))['image_id']
)
df_all = df_all[~df_all['image_id'].isin(hard_ids)].reset_index(drop=True)

# Fold 3 → meta-treino (nunca visto pelos backbones durante treino)
df_meta = df_all[df_all['fold'] == 3].reset_index(drop=True)

print(f'Meta-treino (fold 3): {len(df_meta)} amostras')
print(f'Teste         : {len(df_test)} amostras')
print(f'Distribuição ISUP (meta):\n{df_meta["isup_grade"].value_counts().sort_index()}')

In [ ]:
def filter_existing_patch(df: pd.DataFrame, images_dir: str) -> pd.DataFrame:
    mask = df['image_id'].apply(
        lambda x: os.path.isfile(os.path.join(images_dir, f'{x}.png'))
    )
    return df[mask].reset_index(drop=True)


df_meta_patch = filter_existing_patch(df_meta, IMAGES_DIR)
df_test_patch = filter_existing_patch(df_test, IMAGES_DIR)

print(f'Meta-treino (patch disponível): {len(df_meta_patch)}')
print(f'Teste       (patch disponível): {len(df_test_patch)}')

## 6. Funções de Inferência

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

import albumentations as Albu

# PandasDataset already does np.transpose(image, (2,0,1)) after transforms,
# so do NOT include ToTensorV2 here — the DataLoader collate_fn converts to tensor.
EVAL_TRANSFORM = Albu.Compose([
    Albu.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


@torch.no_grad()
def get_patch_probs(
    model, df: pd.DataFrame, images_dir: str,
    batch_size: int, device, desc: str = 'Patch',
) -> tuple[np.ndarray, np.ndarray, list]:
    """Inferência patch-level → (N,5) probabilidades sigmoidais."""
    ds = PandasDataset(images_dir, df, transforms=EVAL_TRANSFORM, format='png')
    dl = DataLoader(ds, batch_size=batch_size, num_workers=NUM_WORKERS,
                    sampler=SequentialSampler(ds), pin_memory=True)
    probs_list, targets_list, ids_list = [], [], []
    model.eval()
    for batch_x, batch_y, batch_ids in tqdm(dl, desc=desc, leave=False):
        logits = model(batch_x.to(device))
        probs  = torch.sigmoid(logits).cpu().numpy()
        probs_list.append(probs)
        targets_list.extend(batch_y.sum(1).long().tolist())
        ids_list.extend([str(x) for x in batch_ids])
    return np.vstack(probs_list), np.array(targets_list), ids_list


def decode_ordinal(probs: np.ndarray) -> np.ndarray:
    """(N,5) sigmoid probs → classe ISUP 0-5."""
    return (probs > 0.5).sum(axis=1)

## 7. Coleta de Probabilidades

In [ ]:
print('══ META-TREINO (fold 3) ══════════════════════════════')

meta_v2s_opt_probs, meta_targets, meta_ids = get_patch_probs(
    model_v2s_opt, df_meta_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'V2S+Optuna  meta')

meta_b0_probs, *_ = get_patch_probs(
    model_b0, df_meta_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'B0  meta')

meta_swa_probs, *_ = get_patch_probs(
    model_swa, df_meta_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'B0+SWA  meta')

meta_v2s_plain_probs, *_ = get_patch_probs(
    model_v2s_plain, df_meta_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'V2S-plain  meta')

meta_b3_probs, *_ = get_patch_probs(
    model_b3, df_meta_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'B3  meta')

print(f'\nForma das probs (meta): {meta_v2s_opt_probs.shape}')
print(f'Targets meta — distribuição: {np.unique(meta_targets, return_counts=True)}')

In [ ]:
print('══ TESTE ════════════════════════════════════════════')

test_v2s_opt_probs, test_targets, test_ids = get_patch_probs(
    model_v2s_opt, df_test_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'V2S+Optuna  test')

test_b0_probs, *_ = get_patch_probs(
    model_b0, df_test_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'B0  test')

test_swa_probs, *_ = get_patch_probs(
    model_swa, df_test_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'B0+SWA  test')

test_v2s_plain_probs, *_ = get_patch_probs(
    model_v2s_plain, df_test_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'V2S-plain  test')

test_b3_probs, *_ = get_patch_probs(
    model_b3, df_test_patch, IMAGES_DIR, BATCH_SIZE, DEVICE, 'B3  test')

print(f'\nForma das probs (teste): {test_v2s_opt_probs.shape}')

# Libera GPU — backbones não são mais necessários
for m in [model_v2s_opt, model_b0, model_swa, model_v2s_plain, model_b3]:
    m.cpu()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('GPU liberada.')

## 8. Feature Matrix para o Meta-Modelo

Cada modelo contribui com 5 probabilidades ordinais → X tem 25 features (5 modelos × 5 limiares).

In [ ]:
MODEL_NAMES  = ['V2S+Optuna', 'B0', 'B0+SWA', 'V2S-plain', 'B3']
FEATURE_COLS = [
    f'{name}_t{i}' for name in MODEL_NAMES for i in range(OUTPUT_CLASSES)
]

# Meta-treino (fold 3)
X_meta = np.hstack([
    meta_v2s_opt_probs,
    meta_b0_probs,
    meta_swa_probs,
    meta_v2s_plain_probs,
    meta_b3_probs,
])
y_meta = meta_targets

# Conjunto de teste
X_test = np.hstack([
    test_v2s_opt_probs,
    test_b0_probs,
    test_swa_probs,
    test_v2s_plain_probs,
    test_b3_probs,
])
y_test = test_targets

print(f'X_meta : {X_meta.shape}   y_meta : {y_meta.shape}')
print(f'X_test : {X_test.shape}   y_test : {y_test.shape}')

# Desempenho individual no meta (referência)
print('\n── Desempenho individual na meta-partição (fold 3):')
for name, probs in zip(MODEL_NAMES,
                       [meta_v2s_opt_probs, meta_b0_probs, meta_swa_probs,
                        meta_v2s_plain_probs, meta_b3_probs]):
    preds = decode_ordinal(probs)
    k = cohen_kappa_score(y_meta, preds, weights='quadratic')
    a = accuracy_score(y_meta, preds)
    print(f'  {name:<12}  QWK={k:.4f}  Acc={a*100:.2f}%')

## 9. Baseline: Ensemble por Média Simples

In [ ]:
def bootstrap_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                      n: int = N_BOOT, seed: int = SEED) -> dict:
    """Retorna {acc, kappa, f1, recall, precision} com média, std, IC 2.5-97.5."""
    rng   = np.random.default_rng(seed)
    n_obs = len(y_true)
    keys  = ['acc', 'kappa', 'f1', 'recall', 'precision']
    arrs  = {k: np.empty(n) for k in keys}
    for i in range(n):
        idx = rng.integers(0, n_obs, size=n_obs)
        arrs['acc'][i]       = accuracy_score(y_true[idx], y_pred[idx])
        arrs['kappa'][i]     = cohen_kappa_score(y_true[idx], y_pred[idx], weights='quadratic')
        arrs['f1'][i]        = f1_score(y_true[idx], y_pred[idx], average='macro', zero_division=0)
        arrs['recall'][i]    = recall_score(y_true[idx], y_pred[idx], average='macro', zero_division=0)
        arrs['precision'][i] = precision_score(y_true[idx], y_pred[idx], average='macro', zero_division=0)
    out = {}
    for k, arr in arrs.items():
        out[k] = {
            'mean': arr.mean(),
            'std' : arr.std(ddof=1),
            'ci_5': np.percentile(arr, 2.5),
            'ci_95': np.percentile(arr, 97.5),
        }
    return out


def print_metrics(name: str, m: dict):
    print(f'[{name}]')
    print(f"  Accuracy : {m['acc']['mean']*100:.2f}% ± {m['acc']['std']*100:.2f}%  "
          f"[{m['acc']['ci_5']*100:.2f}% – {m['acc']['ci_95']*100:.2f}%]")
    print(f"  QW Kappa : {m['kappa']['mean']:.4f} ± {m['kappa']['std']:.4f}  "
          f"[{m['kappa']['ci_5']:.4f} – {m['kappa']['ci_95']:.4f}]")
    print(f"  Macro F1 : {m['f1']['mean']:.4f} ± {m['f1']['std']:.4f}  "
          f"[{m['f1']['ci_5']:.4f} – {m['f1']['ci_95']:.4f}]")


# Baseline: média simples dos 5 modelos
preds_mean = decode_ordinal(X_test[:, :5*OUTPUT_CLASSES].reshape(-1, 5, OUTPUT_CLASSES).mean(axis=1))
metrics_mean = bootstrap_metrics(y_test, preds_mean)
print_metrics('Baseline — Média Simples (5 modelos)', metrics_mean)

## 10. Treinamento dos Meta-Modelos

Para estimativa OOF robusta no meta-treino, usamos `StratifiedKFold(k=5)` dentro do fold 3.

In [ ]:
def meta_cv_evaluate(
    name: str,
    clf,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test_data: np.ndarray,
    y_test_data: np.ndarray,
    n_splits: int = META_CV_FOLDS,
    scale: bool = False,
) -> dict:
    """
    1. Estimativa OOF no meta-treino via StratifiedKFold.
    2. Retreina com todos os dados de treino.
    3. Prediz no teste e calcula bootstrap CI.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    # ── OOF no meta-treino ────────────────────────────────────────────
    oof_preds = np.zeros(len(y_train), dtype=int)
    oof_proba = np.zeros((len(y_train), 6))
    test_proba_folds = np.zeros((len(y_test_data), 6))

    for fold_idx, (tr_idx, va_idx) in enumerate(
            skf.split(X_train, y_train)):
        X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
        X_va, y_va = X_train[va_idx], y_train[va_idx]

        if scale:
            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_va = sc.transform(X_va)
            X_te = sc.transform(X_test_data)
        else:
            X_te = X_test_data

        clf.fit(X_tr, y_tr)
        oof_preds[va_idx]  = clf.predict(X_va)
        oof_proba[va_idx]  = clf.predict_proba(X_va)
        test_proba_folds  += clf.predict_proba(X_te) / n_splits

    oof_kappa = cohen_kappa_score(y_train, oof_preds, weights='quadratic')
    oof_acc   = accuracy_score(y_train, oof_preds)
    print(f'[{name}] OOF (meta-treino, {n_splits}-fold)  '
          f'QWK={oof_kappa:.4f}  Acc={oof_acc*100:.2f}%')

    # ── Predição final no teste (média dos folds) ─────────────────────
    test_preds = test_proba_folds.argmax(axis=1)
    metrics    = bootstrap_metrics(y_test_data, test_preds)
    print_metrics(f'{name} → TESTE', metrics)

    return {
        'name'       : name,
        'oof_kappa'  : oof_kappa,
        'oof_acc'    : oof_acc,
        'test_preds' : test_preds,
        'test_proba' : test_proba_folds,
        'metrics'    : metrics,
    }

### 10.1 Regressão Logística

In [ ]:
results = {}

# ── Logistic Regression (C=1.0) ───────────────────────────────────────
lr_1 = LogisticRegression(
    C=1.0, max_iter=2000, multi_class='auto',
    solver='lbfgs', random_state=SEED,
)
results['LR-C1.0'] = meta_cv_evaluate(
    'LR-C1.0', lr_1, X_meta, y_meta, X_test, y_test, scale=True)

print()

# ── Logistic Regression (C=0.1 — mais regularizado) ───────────────────
lr_01 = LogisticRegression(
    C=0.1, max_iter=2000, multi_class='auto',
    solver='lbfgs', random_state=SEED,
)
results['LR-C0.1'] = meta_cv_evaluate(
    'LR-C0.1', lr_01, X_meta, y_meta, X_test, y_test, scale=True)

print()

# ── Logistic Regression (C=10.0 — menos regularizado) ─────────────────
lr_10 = LogisticRegression(
    C=10.0, max_iter=2000, multi_class='auto',
    solver='lbfgs', random_state=SEED,
)
results['LR-C10.0'] = meta_cv_evaluate(
    'LR-C10.0', lr_10, X_meta, y_meta, X_test, y_test, scale=True)

### 10.2 Random Forest

In [ ]:
# ── Random Forest (padrão) ─────────────────────────────────────────────
rf_default = RandomForestClassifier(
    n_estimators=300, max_depth=5,
    max_features='sqrt', min_samples_leaf=2,
    n_jobs=-1, random_state=SEED,
)
results['RF-d5'] = meta_cv_evaluate(
    'RF-d5', rf_default, X_meta, y_meta, X_test, y_test, scale=False)

print()

# ── Random Forest (profundidade maior) ───────────────────────────────
rf_deep = RandomForestClassifier(
    n_estimators=300, max_depth=10,
    max_features='sqrt', min_samples_leaf=1,
    n_jobs=-1, random_state=SEED,
)
results['RF-d10'] = meta_cv_evaluate(
    'RF-d10', rf_deep, X_meta, y_meta, X_test, y_test, scale=False)

print()

# ── Random Forest (sem limite de profundidade) ────────────────────────
rf_full = RandomForestClassifier(
    n_estimators=300, max_depth=None,
    max_features='sqrt', min_samples_leaf=1,
    n_jobs=-1, random_state=SEED,
)
results['RF-full'] = meta_cv_evaluate(
    'RF-full', rf_full, X_meta, y_meta, X_test, y_test, scale=False)

## 11. Tabela Comparativa de Resultados

In [ ]:
rows = []

# Baselines históricos (referência para contexto)
HIST = {
    'V2S+Optuna (individual)' : {'kappa': 0.8742, 'acc': 71.51, 'f1': 0.6772},
    'B0+SWA (individual)'     : {'kappa': 0.8523, 'acc': 65.35, 'f1': 0.6123},
    'Six-Weighted-Mean'        : {'kappa': 0.8810, 'acc': 72.72, 'f1': 0.6787},
    'Ensemble-Weighted (B0+B3+B7)': {'kappa': 0.8847, 'acc': 72.20, 'f1': 0.6850},
}
for name, v in HIST.items():
    rows.append({
        'Modelo': name, 'Tipo': 'histórico',
        'OOF QWK': '—',
        'Acc (%)': v['acc'], 'QWK': v['kappa'], 'F1-Macro': v['f1'],
        'QWK CI 5%': '—', 'QWK CI 95%': '—',
    })

# Baseline média simples top-5
m = metrics_mean
rows.append({
    'Modelo': 'Média Simples (top-5 patch)', 'Tipo': 'baseline',
    'OOF QWK': '—',
    'Acc (%)': round(m['acc']['mean']*100, 2),
    'QWK': round(m['kappa']['mean'], 4),
    'F1-Macro': round(m['f1']['mean'], 4),
    'QWK CI 5%': round(m['kappa']['ci_5'], 4),
    'QWK CI 95%': round(m['kappa']['ci_95'], 4),
})

# Meta-modelos
for key, res in results.items():
    m = res['metrics']
    rows.append({
        'Modelo': key, 'Tipo': 'stacking',
        'OOF QWK': round(res['oof_kappa'], 4),
        'Acc (%)': round(m['acc']['mean']*100, 2),
        'QWK': round(m['kappa']['mean'], 4),
        'F1-Macro': round(m['f1']['mean'], 4),
        'QWK CI 5%': round(m['kappa']['ci_5'], 4),
        'QWK CI 95%': round(m['kappa']['ci_95'], 4),
    })

df_cmp = pd.DataFrame(rows)
df_stacking = df_cmp[df_cmp['Tipo'] == 'stacking'].copy()
df_cmp_sorted = pd.concat([
    df_cmp[df_cmp['Tipo'] == 'histórico'],
    df_cmp[df_cmp['Tipo'] == 'baseline'],
    df_stacking.sort_values('QWK', ascending=False),
]).reset_index(drop=True)

print('\n' + '='*90)
print(df_cmp_sorted[['Modelo', 'Tipo', 'OOF QWK', 'Acc (%)', 'QWK', 'F1-Macro',
                      'QWK CI 5%', 'QWK CI 95%']].to_string(index=False))
print('='*90)

best_stack = df_stacking.sort_values('QWK', ascending=False).iloc[0]
print(f"\n★ Melhor meta-modelo: {best_stack['Modelo']}  QWK={best_stack['QWK']:.4f}")

## 12. Visualizações

In [ ]:
# Gráfico de barras: QWK por estratégia
df_plot = df_cmp_sorted[df_cmp_sorted['QWK'] != '—'].copy()
df_plot['QWK'] = df_plot['QWK'].astype(float)

palette = {'histórico': '#95a5a6', 'baseline': '#3498db', 'stacking': '#e74c3c'}
colors  = [palette[t] for t in df_plot['Tipo']]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(df_plot['Modelo'], df_plot['QWK'],
               color=colors, edgecolor='white', height=0.7)
for bar, v in zip(bars, df_plot['QWK']):
    ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=8)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for t, c in palette.items()]
ax.legend(handles=legend_elements, loc='lower right')
ax.set_xlabel('Quadratic Weighted Kappa (QWK)')
ax.set_title('Stacking Top-5 vs. Baselines Históricos')
ax.axvline(0.87, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.set_xlim(df_plot['QWK'].min() - 0.02, df_plot['QWK'].max() + 0.02)
plt.tight_layout()
plt.savefig('logs/stacking-top5-comparison.png', dpi=200)
plt.show()

In [ ]:
# Matrizes de confusão: melhor LR, melhor RF e Média Simples
best_lr_key  = df_stacking[df_stacking['Modelo'].str.startswith('LR')].sort_values('QWK', ascending=False).iloc[0]['Modelo']
best_rf_key  = df_stacking[df_stacking['Modelo'].str.startswith('RF')].sort_values('QWK', ascending=False).iloc[0]['Modelo']

labels = [f'ISUP {i}' for i in range(6)]
configs = [
    ('Média Simples', preds_mean),
    (best_lr_key, results[best_lr_key]['test_preds']),
    (best_rf_key, results[best_rf_key]['test_preds']),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, preds) in zip(axes, configs):
    cm_norm = confusion_matrix(y_test, preds, normalize='true')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax,
                vmin=0, vmax=1)
    kappa = cohen_kappa_score(y_test, preds, weights='quadratic')
    ax.set_title(f'{title}\nQWK={kappa:.4f}', fontsize=10)
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices (normalizada por linha)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('logs/stacking-top5-confusion-matrices.png', dpi=200)
plt.show()

In [ ]:
# Feature importance — Random Forest melhor
# Retreina RF com todos os dados de meta-treino para obter importâncias
best_rf_clf = results[best_rf_key]

rf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=int(best_rf_key.split('-d')[-1]) if 'd' in best_rf_key else None,
    max_features='sqrt', min_samples_leaf=1,
    n_jobs=-1, random_state=SEED,
)
rf_final.fit(X_meta, y_meta)

importances = pd.Series(rf_final.feature_importances_, index=FEATURE_COLS)
importances_grouped = importances.groupby(
    lambda col: col.rsplit('_', 1)[0]
).sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por modelo
importances_grouped.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Importância por Modelo (RF)')
axes[0].set_xlabel('Importância total')

# Por limiar (threshold)
importances_by_threshold = importances.groupby(
    lambda col: f't{col.rsplit("_t", 1)[-1]}'
).sum().sort_values(ascending=True)
importances_by_threshold.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Importância por Limiar Ordinal (RF)')
axes[1].set_xlabel('Importância total')

plt.suptitle(f'Feature Importance — {best_rf_key}', fontsize=12)
plt.tight_layout()
plt.savefig('logs/stacking-top5-feature-importance.png', dpi=200)
plt.show()

print('\nImportância por modelo:')
print(importances_grouped.sort_values(ascending=False).to_string())

## 13. Relatório Detalhado do Melhor Meta-Modelo

In [ ]:
best_key    = best_stack['Modelo']
best_preds  = results[best_key]['test_preds']
best_metrics = results[best_key]['metrics']

print('=' * 70)
print(f'MELHOR META-MODELO: {best_key}')
print('=' * 70)
print_metrics(best_key, best_metrics)
print()
print(classification_report(y_test, best_preds,
                             target_names=[f'ISUP {i}' for i in range(6)]))

## 14. Salvar Resultados

In [ ]:
# CSV com predições de todos os meta-modelos
save_df = pd.DataFrame({
    'image_id'  : test_ids,
    'true_label': y_test,
    'pred_mean' : preds_mean,
    **{f'pred_{key}': res['test_preds'] for key, res in results.items()},
})
save_df.to_csv('logs/stacking-top5-predictions.csv', index=False)

# Relatório TXT
report_path = 'logs/stacking-top5-results.txt'
with open(report_path, 'w') as f:
    f.write('Stacking — Top-5 Modelos Individuais\n')
    f.write('=' * 70 + '\n\n')
    f.write('Bootstrap resamples: 1000\n\n')
    f.write(f'{"Modelo":<22} {"OOF QWK":>10} {"Acc (%)":>10} {"QWK":>8} {"F1":>8} {"QWK CI 5%":>12} {"QWK CI 95%":>12}\n')
    f.write('-' * 90 + '\n')
    for _, row in df_cmp_sorted.iterrows():
        f.write(
            f"{str(row['Modelo']):<22} "
            f"{str(row['OOF QWK']):>10} "
            f"{str(row['Acc (%)']):>10} "
            f"{str(row['QWK']):>8} "
            f"{str(row['F1-Macro']):>8} "
            f"{str(row['QWK CI 5%']):>12} "
            f"{str(row['QWK CI 95%']):>12}\n"
        )
    f.write('\n' + '=' * 70 + '\n')
    f.write(f'Melhor meta-modelo: {best_key}  QWK={best_stack["QWK"]:.4f}\n')
    f.write('\nClassification Report (melhor):\n')
    f.write(classification_report(y_test, best_preds,
                                   target_names=[f'ISUP {i}' for i in range(6)]))

print(f'Resultados salvos em {report_path}')
print(f'Predições salvas em logs/stacking-top5-predictions.csv')